# 🧱 LLM Lego

---

### Moduli

```
testo  →  pezzi di un vocabolario finito  →  vettore  →  algebra
          Modulo 1                            Modulo 2    Modulo 3
```


---
## Setup

La cella qui sotto installa tutto quello che serve e scarica GloVe (~130MB).
La durata per il setup è di circa 2-3 minuti.


In [ ]:
# Clona il repo della lezione e installa le dipendenze
!git clone -q -b scuole https://github.com/tlm-journalclub-org/LLMlego.git
%cd LLMlego
!python setup_colab.py

# Abilita i widget interattivi in Colab (va eseguito DAL kernel del notebook,
# non da setup_colab.py che gira come subprocess)
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # non siamo in Colab

# Importa la libreria didattica
from llmlego_scuola import *
import numpy as np
print("Tutto pronto!")


---
# Modulo 1: Dal testo a un vocabolario finito

Un LLM ragiona su un **vocabolario finito** di pezzi di testo. Senza questo limite, dovrebbe rappresentare un numero grandissimo di parole (alcune rarissime) nello spazio dei vettori.

Quindi il primissimo passo è: dato un testo, **spezzarlo in pezzi che appartengono a un vocabolario fisso**.

Partiamo da un piccolo **corpus**.


In [ ]:
# da Wikipedia
corpus = [
    "Un LLM è un tipo di modello linguistico notevole per essere in grado di ottenere la comprensione e la generazione di linguaggio di ambito generale.",
    "Gli LLM acquisiscono questa capacità adoperando enormi quantità di dati per apprendere miliardi di parametri nell'addestramento e consumando grandi risorse di calcolo nell'operatività.",
    "L'aggettivo grande presente nel nome si riferisce all'ingente quantità di parametri del modello probabilistico (nell'ordine dei miliardi)"
]


### Il vocabolario del corpus

Per usare questo corpus, costruiamo (lo facciamo noi qui sotto) il **vocabolario** delle parole uniche. Il vocabolario è come se fosse un catalogo delle parole che il nostro modello conosce, e ogni parola ha un posto preciso in questo catalogo (un numero identificativo).

> 📦 Grazie alla funzione `mostra_dizionario(vocabolario)` visualizziamo il dizionario come tabella ordinata: per ogni parola unica del corpus c'è il suo ID intero.


In [ ]:
# Costruiamo il vocabolario: ogni parola unica ottiene un posto
vocabolario = {}
for frase in corpus:
    for parola in frase.split():
        if parola not in vocabolario:
            vocabolario[parola] = len(vocabolario)

mostra_dizionario(vocabolario)

### Il problema delle parole nuove

Funziona se le parole stanno nel nostro vocabolario. **Ma se ne arriva una nuova?**

Provate a immaginare cosa succede se passiamo *"Gli LLM sono in larga parte reti neurali e in particolare Transformers"*: `Transformers` e `reti neurali` non sono nel vocabolario.

Un LLM vero come GPT ha questo stesso problema, ma adotta una soluzione geniale: **spezza le parole sconosciute in pezzi più piccoli (subword token)**. Il suo vocabolario contiene ~100.000 pezzi e con questi può comporre praticamente **qualunque** parola, anche quelle che non ha mai visto.

> 📦 Grazie alla funzione `mostra_token("una frase")` viene mostrato come la frase viene tokenizzata dal tokenizer di GPT-4 (`cl100k_base`): ogni token ha un colore diverso e il suo ID intero nel vocabolario. Il simbolo `·` rappresenta uno **spazio** che fa parte del token che segue (in BPE lo spazio è solitamente attaccato alla parola successiva: il chip `·gatto` significa "spazio + gatto").


In [ ]:
mostra_token("Gli LLM sono in larga parte reti neurali e in particolare Transformers")

In [ ]:
# Confronto italiano vs inglese: la stessa frase, due lingue
mostra_token("Il gatto è sul tappeto")
mostra_token("The cat sat on the mat")


In [ ]:
# Parole "inventate" e numeri grandi
mostra_token("supercalifragilistichespiralidoso")
mostra_token("1234567890")

### Sintesi

- Un LLM ha un **vocabolario finito di pezzi (token)** (~100.000 in GPT-4).
- Le parole sconosciute vengono **spezzate** in pezzi noti.
- L'**italiano** usa più token dell'inglese per dire la stessa cosa (la maggior parte dei modelli sono allenati su corpus in maggioranza inglesi).

Ora abbiamo un vocabolario finito di parole. Per aggiungere le **relazioni di significato** tra le parole manca un passaggio: trasformare le parole in numeri, o meglio in **vettori**.

---
# Modulo 2: Le parole come vettori, esempio in 2D

Per rappresentare una parola scegliamo delle **caratteristiche (features)**. Ogni feature è una **dimensione** dello spazio.
Per ora ne usiamo 2 (così le vediamo sul piano cartesiano):

- **asse X = grado reale**: quanto la parola indica una "persona reale" (re, regina, principe, ...)
- **asse Y = genere**: quanto la parola è "femminile"

Più avanti vedremo che in un modello vero ci sono **molte più feature** (per esempio genere, dimensione, mammifero, paese, ...) ma il principio è identico.

> **Nota tecnica**: per costruzione, nel nostro spazio inventato ogni componente dei vettori vive tra **−1** e **+1**. In questo modo i valori delle feature sono "normalizzati", −1 è il minimo e +1 il massimo.

> 📦 Grazie alla funzione `plotta_2d(spazio)` le parole vengono disegnate come punti nel piano cartesiano, ognuna con una freccia che parte dall'origine fino al suo punto.

Definiamo a mano le coordinate per 6 parole:


In [ ]:
spazio_2d = { # (regale, femminile)
    "re":         ( 0.9, -0.7),
    "regina":     ( 0.9,  0.7),
    "uomo":       (-0.5, -0.7),
    "donna":      (-0.5,  0.7),
    "principe":   ( 0.6, -0.6),
    "principessa":( 0.6,  0.6),
}

plotta_2d(spazio_2d)

### La "fotografia" del vettore

Possiamo anche visualizzare un singolo vettore come una sequenza di celle colorate, una per dimensione.
Blu = positivo, rosso = negativo, intensità = grandezza del valore.

> 📦 La funzione `mostra_vettore_2d(parola, spazio)` mostra le 2 componenti del vettore della parola come 2 celle colorate, con la stessa logica che useremo poi per i veri vettori a 100 dimensioni.


In [ ]:
mostra_vettore_2d("re", spazio_2d)
mostra_vettore_2d("regina", spazio_2d)


In questo modo il nostro spazio ci permette di rappresentare non solo delle parole ma anche delle relazioni di significato tra le parole: in base a dove si trova la parola nello spazio essa ha delle caratteristiche. Parole vicine nello spazio avranno caratteristiche simili, parole lontane avranno caratteristiche diverse.

### Misurare la vicinanza: la cosine similarity

In uno spazio vettoriale, due parole sono "simili" se i loro vettori puntano **nella stessa direzione**. Per misurare quanto due vettori puntano nella stessa direzione si usa la **cosine similarity**:

$$\cos\theta = \frac{\vec a \cdot \vec b}{\|\vec a\| \cdot \|\vec b\|}$$

dove $\theta$ è l'angolo compreso tra $\vec a$ e $\vec b$. Vale sempre tra **−1 e +1**:

- $\cos\theta = 1$ → stessa direzione (similarità massima)
- $\cos\theta = 0$ → vettori perpendicolari
- $\cos\theta = -1$ → direzioni opposte

> 📦 `mostra_similarita_2d(spazio, parola_a, parola_b)` disegna i due vettori dall'origine, l'angolo θ tra loro e il valore di cos(θ).

Vediamolo geometricamente sul nostro spazio inventato. Confrontiamo prima `principessa` con `regina` (vicini), poi `re` con `donna` (lontani):


In [ ]:
# Due parole "vicine" → angolo piccolo, cos(θ) alta
mostra_similarita_2d(spazio_2d, "principessa", "regina")

In [ ]:
# Due parole "lontane" (diverso genere E diverso grado reale) → angolo grande
mostra_similarita_2d(spazio_2d, "re", "donna")


> 📦 La funzione `tabella_similarita_2d(spazio, coppie)` confronta tante coppie di parole in colpo solo: ogni riga mostra la cosine similarity tra le due parole e una barra divergente (blu verso destra = positivo, rossa verso sinistra = negativo). Useremo la stessa convenzione anche più avanti per visualizzare il bias.


In [ ]:
# Tabella della similarità su più coppie
tabella_similarita_2d(spazio_2d, [
    ("re", "regina"),
    ("re", "principe"),
    ("re", "uomo"),
    ("re", "donna"),
    ("uomo", "donna"),
])


### Due principi chiave dello spazio vettoriale

Da questo momento in poi, qualunque sia il modello vero, ricordiamoci sempre questi due principi:

1. **Vicinanza = similarità di significato.** Parole con cos(θ) alta hanno significati legati.
2. **Direzione (differenza) tra due vettori = sfumatura di senso.** Le frecce nello spazio (es. da `uomo` a `donna`) codificano relazioni semantiche.

Ora vediamo che la seconda osservazione ha una conseguenza algebrica.

### Esercizio 2.1: Aritmetica su carta

Guardate il piano. **Cosa vi aspettate** se fate:

$$\vec{re} - \vec{uomo} + \vec{donna} = \;?$$

Calcolatelo a mano e disegnate il risultato.


In [ ]:
# Trasforma in array numpy per fare aritmetica vettoriale
v_re    = np.array(spazio_2d["re"])
v_uomo  = np.array(spazio_2d["uomo"])
v_donna = np.array(spazio_2d["donna"])

risultato = v_re - v_uomo + v_donna
print(f"Risultato: {risultato}")


In [ ]:
# Visualizziamolo nello spazio (in rosso)
plotta_2d(spazio_2d, evidenzia={"re - uomo + donna": tuple(risultato)})


### Sintesi:

La **relazione "maschile → femminile"** è codificata come una **direzione** nello spazio.
Posso applicarla a qualunque parola.


### Esercizio 2.2: La direzione femminile

Costruisci il **vettore-direzione** `femminile = donna − uomo`, poi applicalo a `principe`.
Dove finisce?


In [ ]:
direzione_femminile = v_donna - v_uomo

v_principe = np.array(spazio_2d["principe"])
principe_femminilizzato = v_principe + direzione_femminile

print(f"Principe + direzione femminile = {principe_femminilizzato}")

plotta_2d(
    spazio_2d,
    evidenzia={"principe + femminile": tuple(principe_femminilizzato)},
)


### La grande domanda

Funziona perché abbiamo scelto noi le coordinate. Ma:

> **Può un computer imparare da solo dove mettere le parole, leggendo testi?**

La risposta è sì, **ma gli servono molte più dimensioni per rappresentare più sfumature possibili, per esempio ~100**. Vediamole.


---
# Modulo 3: Embedding veri (100 dimensioni)

**GloVe** è un modello che ha letto miliardi di parole inglesi (Wikipedia, news) e ha imparato
a posizionare ogni parola in uno spazio a **100 dimensioni**.

Ogni dimensione non ha una feature leggibile come "regalità" o "femminilità": è più complicato interpretarle, sono dimensioni emergenti scoperte dal modello.

**Quello che conta è che l'algebra dei vettori però funziona allo stesso modo.** Quello che abbiamo fatto in 2D lo rifacciamo in 100D.

> 🇮🇹 -> 🇬🇧 Da qui in poi lavoriamo in **inglese**: i modelli più studiati sono in inglese
> perché c'è molto più testo disponibile per allenarli.

> 📦 La funzione `mostra_vettore("parola")` è la versione 100D di `mostra_vettore_2d`: invece di 2 celle ne mostra 100, una per dimensione del vettore di GloVe.


In [ ]:
# Vediamo com'è fatto il vettore della parola "king"
mostra_vettore("king")

Sopra c'è una "fotografia" del vettore di `king`: 100 dimensioni, alcune positive (blu) e alcune negative (rosso).
Nessuna ha un significato esplicito singolarmente: è la **combinazione**, ovvero la regione dello spazio dove si trova il vettore, che codifica il significato.

> 📦 Con `widget_vicini("parola")` apriamo un campo di testo interattivo: man mano che cambi la parola, vedi aggiornarsi la lista dei suoi vicini più simili nello spazio (calcolati con cosine similarity sulla top-30k del vocabolario di GloVe).


In [ ]:
# Esplora i vicini di una parola: digita una parola inglese qui sotto
widget_vicini("king")

### Esercizio 3.1: `king − man + woman = ?`

Costruisci il vettore e cerca la parola più vicina.

> 📦 La funzione `vettore("parola")` ritorna il vettore (array di 100 numeri) di una parola di GloVe. Possiamo poi sommare e sottrarre questi vettori con numpy. Infine `parola_piu_vicina(v)` cerca nello spazio la parola del vocabolario il cui vettore è più simile (cosine similarity più alta) al risultato.


In [ ]:
v = vettore("king") - vettore("man") + vettore("woman")
risultato = parola_piu_vicina(v, escludi="king")
print(f"king - man + woman ≈ {risultato}")

### Esercizio 3.2: La direzione del genere

Costruisci il vettore-direzione `genere = woman − man` e applicalo ad altre parole.


In [ ]:
direzione_genere = vettore("woman") - vettore("man")

# Applichiamola ad "actor": ci aspettiamo "actress"
nuovo = vettore("actor") + direzione_genere
print(f"actor + (woman - man) ≈ {parola_piu_vicina(nuovo, escludi='actor')}")


### Esercizio 3.3: Prova tu, 3 analogie

Completa le 3 analogie.


In [ ]:
# Analogia 1: capitale -> paese, geografia
v = vettore("paris") - vettore("france") + vettore("germany")
print("paris - france + germany ≈", parola_piu_vicina(v, escludi="paris"))

# Analogia 2: tempo verbale
v = vettore("walking") - vettore("walk") + vettore("swim")
print("walking - walk + swim ≈", parola_piu_vicina(v, escludi="swim"))

# Analogia 3: inventane una tua! Sostituisci le parole qui sotto:
v = vettore("...") - vettore("...") + vettore("...")
# print("la tua analogia ≈", parola_piu_vicina(v, escludi="..."))


---
# Modulo 4: Word Golf

### Regole

1. Vi diamo una **parola di partenza** e una **parola obiettivo (target)**.
2. A ogni mossa scegliete `g.aggiungi("parola")` oppure `g.sottrai("parola")` da una lista.
3. Dopo ogni mossa, la vostra **nuova parola corrente** è quella più vicina al vettore risultante.
4. **Vincete** quando il `target` compare nei **top-5 vicini** del risultato di una mossa.
5. **Vince la coppia con meno mosse o che impiega meno tempo a parità di mosse.**


### Prima di iniziare: nome della squadra

Inserite un **nome di squadra**.

In [ ]:
# 1) Scegliete il nome della vostra squadra
NOME_SQUADRA = "SQUADRA A"   # <-- CAMBIATE QUESTO

# Collega tutto: se DASHBOARD_URL è settato, ogni vittoria viene inviata alla dashboard
import llmlego_scuola.golf as _golf
DASHBOARD_URL = _golf.DASHBOARD_URL


### Le parole operatore

Le mosse del Word Golf si fanno aggiungendo/sottraendo parole prese da un set fisso (~60 parole, divise per categorie).

> 📦 `mostra_parole_operatore()` stampa la griglia di queste parole organizzate per categoria (persone, luoghi, cibo, oggetti, natura, tempo, astratti).


In [ ]:
# Queste sono le ~60 parole disponibili come operatori:
mostra_parole_operatore()

### Come si usa: la classe `WordGolf`

Prima di partire coi round veri, vediamo come funziona il gioco con un mini-esempio guidato dal prof.

> 📦 La classe `WordGolf(start, target, squadra)` crea una partita. Sull'oggetto creato potete usare:
> - `.aggiungi("parola")` → somma il vettore della parola al vettore della parola corrente, snappa alla parola del vocabolario più vicina
> - `.sottrai("parola")` → analoga ma con la sottrazione
> - `.stato()` → mostra le mosse fatte finora, con i top-5 vicini ad ogni passo
>
> Si vince quando il `target` compare nei top-5 vicini del risultato.


In [ ]:
# DEMO guidata
demo = WordGolf(start="man", target="actress", squadra="demo")

# demo.aggiungi("woman")
# demo.sottrai("boy")
# demo.stato()


## Round 1 (warm-up): `doctor → farmer`

Spostiamo un mestiere a un altro. Quale direzione vi viene in mente?


In [ ]:
g1 = WordGolf(start="doctor", target="farmer", squadra=NOME_SQUADRA)

# Le vostre mosse:



## Round 2: `paris → tokyo`

Geografia. Come si muove una capitale in un'altra?


In [ ]:
g2 = WordGolf(start="paris", target="tokyo", squadra=NOME_SQUADRA)

# Le vostre mosse:


## Round 3: `boy → queen`

Una trasformazione stile favola: cosa deve succedere a un ragazzino per diventare regina?


In [ ]:
g3 = WordGolf(start="boy", target="queen", squadra=NOME_SQUADRA)

# Le vostre mosse:


## Round 5 (bonus): `cat → eagle`

Trasformare un gatto in un'aquila non è banale. Indizio: il gatto deve perdere qualcosa di "domestico" prima di diventare un animale selvatico.


In [ ]:
g5 = WordGolf(start="cat", target="eagle", squadra=NOME_SQUADRA)

# Le vostre mosse:


## 🏆 Sfida finale: `science → dance`

L'ultima sfida è la più astratta: dalla scienza, una disciplina razionale, arrivare alla danza, pura espressione del corpo. La risolverete sottraendo gli aspetti più "rigidi" della scienza e aggiungendo quelli più "espressivi".


In [ ]:
g6 = WordGolf(start="science", target="dance", squadra=NOME_SQUADRA)

# Le vostre mosse:


---
### ⚠️ Le relazioni nei dati: il bias

Se il modello impara la direzione **genere** leggendo testi reali...
**impara anche i bias che ci sono nei testi reali.**

Calcoliamo, per ogni professione, la **cosine similarity** tra il suo vettore e la direzione `woman − man`. I valori vivono tra **−1** (allineato verso *man*) e **+1** (allineato verso *woman*).

> 📦 `mostra_bias_professioni()` fa proprio questo calcolo su una lista di mestieri (doctor, nurse, engineer, ...) e mostra il risultato come bar chart divergente, esattamente con la stessa convenzione blu/rossa che abbiamo visto in `tabella_similarita_2d`.


In [ ]:
mostra_bias_professioni()

**Cosa vedete?** 

Questo NON è un'opinione di GloVe. È lo **specchio** dei testi su cui è stato allenato (Wikipedia, news).
